In [4]:
data_dir = 'document_collection/'

!python ../instructlab/docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../instructlab/docling_v2_config.yaml

2025-09-27 03:39:47,963 - INFO - Found 1 PDF files to process
2025-09-27 03:39:47,967 - INFO - Processing document_collection/bmo.pdf
2025-09-27 03:39:47,967 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-09-27 03:39:47,975 - INFO - Going to convert document batch...
2025-09-27 03:39:47,975 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 6024c602531576fa77a98611ef9b35dd
2025-09-27 03:39:47,980 - INFO - Loading plugin 'docling_defaults'
2025-09-27 03:39:47,980 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-09-27 03:39:47,986 - INFO - Loading plugin 'docling_defaults'
2025-09-27 03:39:47,986 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-09-27 03:39:51,078 - INFO - Accelerator device: 'cuda:0'
2025-09-27 03:39:53,253 - INFO - Accelerator device: 'cuda:0'
2025-09-27 03:40:14,189 - INFO - Accelerator device: 'cuda:0'
2025-09-27 03:40:15,165 - INFO - Processing document bmo.pdf
2025-09-27 

In [ ]:
%pip install docling
%pip install --upgrade transformers
%pip install markdown-it-py

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import glob

with open(glob.glob(f'{data_dir}/*.md')[0], 'r') as f:
    text = f.read()

In [52]:
from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(
    text: str,
    max_tokens: int = 200,
    overlap: int = 50
) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.
    """

    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Now split blocks into chunks with overlap
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # emit a chunk
                chunks.append(" ".join(current_words))
                # prepare next buffer with overlap from *this chunk*
                current_words = current_words[-overlap:] if overlap > 0 else []

    # flush remaining words
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)

seed_data = datasets.Dataset.from_dict({'document': chunks})
icl = {
    "document_outline": "The document contains excerpts from FINTRAC regulations designed to combat money laundering and terrorist financing in Canada",
    "icl_document": "## Overview\n\nThis guidance came into effect on June 1, 2021.\n\n\nThis guidance explains the methods that can be used by reporting entities\n(REs) to verify the identity of a person or an entity.\n\n\n## 1. Meaning of verifying the identity of a person or an entity\n\nIt means to use the methods described in this guidance to ensure that the\ninformation in an identification document or from other informational\nsources matches the information that the person or entity provided.\n\n\nVerifying identity is a foundational element of Canada's anti-money\nlaundering and anti-terrorist financing regime and a key component of an\nRE's relationship with clients. It helps you to know your clients and to\nunderstand and assess any risk that may be associated to their\ntransactions or activities.\n\n\n## 2. How to verify the identity of a person\n\nYou can use any of the 5 methods described below to identify a person:\n\n- 2.1 Government-issued photo identification method\n\n- 2.2 Credit file method\n\n- 2.3 Dual-process method\n\n- 2.4 Affiliate or member method\n\n- 2.5 Reliance method\n",
    "icl_query_1": "In Canada, what are the methods for verifying someone's identity?",
    "icl_query_2": "In Canada, why is it important to confirm a client's identity?",
    "icl_query_3": "In Canada, can I use Reliance method to verify identity of a person?",
}
seed_data = seed_data.map(lambda x: icl)
seed_data.to_json('seed_data.jsonl', orient='records', lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 617.54ba/s]


104830